# MDP Image Recognition — YOLO26 Segmentation with Background Randomization

Recognizes the MDP symbol cards (image IDs **11–40** + bullseye `marker`) from arena photos.

**Model: `yolo26s-seg`** (Ultralytics, Jan 2026 — their recommended production model; +2.5 box AP /
+3.7 mask AP over YOLO11, NMS-free end-to-end inference, ~43% faster CPU inference for edge deploys).
One-line fallback to `yolo11s-seg` if anything misbehaves.

**Why segmentation (not plain detection):** our labels are glyph-outline polygons, and mask
supervision forces the model to learn the *shape of the glyph itself* rather than the card
background — exactly what we need since the real test may put symbols on scattered / noisy /
non-black backgrounds. Segmentation models still output boxes + classes, so inference is unchanged.

**Background robustness strategy** (the real test won't always be white-on-black cards):
1. *Offline card-background randomization* (section 3): keep the glyph pixels via their polygon
   mask, replace the surrounding card area with random textures — scatter dots, noise, stripes,
   random image crops, random colors. Labels stay identical.
2. *Online augmentation*: strong HSV jitter, mosaic, segment-based copy-paste, random erasing.
3. `fliplr=0, flipud=0` — arrows/letters are orientation-sensitive (a mirrored left arrow **is** a
   right arrow; this is the #1 documented pitfall for this task).

See `initial_plan.md` for the full research summary.

In [ ]:
# One-time setup. NOTE: quote the version spec — unquoted `>=` is a shell redirect
# and silently installs the wrong thing (it also breaks NumPy in this NGC container).
%pip install -q "ultralytics>=8.4" "numpy<2"
# This container's torch/cv2/numba are built against NumPy 1.x, so numpy must stay <2.
# If you just ran this for the first time, restart the kernel before continuing.

In [ ]:
from pathlib import Path
import glob, random, collections

import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

ROOT = Path.cwd()  # run this notebook from the mdp/ folder
assert (ROOT / "data.yaml").exists(), f"Run this notebook from the dataset folder, cwd={ROOT}"

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 1. Dataset sanity check

In [ ]:
CLASS_NAMES = ['11','12','13','14','15','16','17','18','19','20','21','22','23','24','25',
               '26','27','28','29','30','31','32','33','34','35','36','37','38','39','40','marker']

# Standard MDP symbol table for the image IDs (verify against your checklist)
SYMBOL = {**{str(10+i): str(i) for i in range(1, 10)},                       # 11-19 -> digits 1-9
          **dict(zip(map(str, range(20, 36)), "ABCDEFGHSTUVWXYZ")),          # 20-35 -> letters
          '36': 'Up arrow', '37': 'Down arrow', '38': 'Right arrow',
          '39': 'Left arrow', '40': 'Stop', 'marker': 'Bullseye'}

for split in ("train", "valid", "test"):
    imgs = sorted((ROOT / split / "images").glob("*.jpg"))
    lbl_dir = ROOT / split / "labels"
    lbls = sorted(lbl_dir.glob("*.txt")) if lbl_dir.exists() else []
    missing = {p.stem for p in imgs} ^ {p.stem for p in lbls}
    status = "OK" if imgs and not missing else f"MISMATCH ({len(missing)} unmatched)"
    print(f"{split:5s}: {len(imgs):5d} images, {len(lbls):5d} labels  {status}")

In [ ]:
# Class distribution across splits
counts = {s: collections.Counter() for s in ("train", "valid", "test")}
for split in counts:
    for f in (ROOT / split / "labels").glob("*.txt"):
        for line in f.read_text().splitlines():
            if line.strip():
                counts[split][CLASS_NAMES[int(line.split()[0])]] += 1

fig, ax = plt.subplots(figsize=(14, 4))
x = np.arange(len(CLASS_NAMES))
for i, (split, c) in enumerate(counts.items()):
    ax.bar(x + (i - 1) * 0.28, [c.get(n, 0) for n in CLASS_NAMES], width=0.28, label=split)
ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES, rotation=45)
ax.set_title("Instances per class"); ax.legend(); plt.tight_layout(); plt.show()

## 2. Visualize samples with their glyph polygons

In [ ]:
def read_polys(lbl_path, w, h):
    out = []
    for line in Path(lbl_path).read_text().splitlines():
        vals = line.split()
        if len(vals) > 5:
            pts = (np.array(vals[1:], dtype=np.float32).reshape(-1, 2) * [w, h]).astype(np.int32)
            out.append((int(vals[0]), pts))
    return out

def show_samples(split="valid", n=6, seed=0):
    rng = random.Random(seed)
    files = rng.sample(sorted((ROOT / split / "labels").glob("*.txt")), n)
    fig, axes = plt.subplots(2, (n + 1) // 2, figsize=(14, 7))
    for ax, lbl in zip(axes.flat, files):
        img = cv2.cvtColor(cv2.imread(str(ROOT / split / "images" / (lbl.stem + ".jpg"))), cv2.COLOR_BGR2RGB)
        titles = []
        for cls_i, pts in read_polys(lbl, img.shape[1], img.shape[0]):
            cv2.polylines(img, [pts], True, (0, 255, 0), 2)
            titles.append(f"{CLASS_NAMES[cls_i]} ({SYMBOL[CLASS_NAMES[cls_i]]})")
        ax.imshow(img); ax.set_title(", ".join(titles), fontsize=9); ax.axis("off")
    plt.tight_layout(); plt.show()

show_samples()

## 3. Card-background randomization (offline synthetic data)

The Roboflow photos are all **bright glyph on a near-black card**. The real arena photo
(`real/`) is the opposite: a **black glyph on a light, finely textured card** — knit, waffle
weave, wood chip, glitter, denim — sitting on dark cloth. A model trained only on the export
will lean on "bright blob on dark square" as its shortcut and fall over on the real thing.

So each labelled card is *repainted* rather than merely having noise pasted behind it:

1. **Recover the real glyph pixels.** The label polygon is a single outline, so filling it also
   fills the counters of `B/8/9/A`; on the arrow classes it traces only the triangular *head*.
   Both are fixed in `glyph_mask_with_holes` (Otsu over glyph+card pixels, plus a capped
   connected-component grow that picks up the arrow shaft).
2. **New card face** — a square-ish quad following the glyph, filled with a procedural
   weave/rib/mottle/glitter texture or a real surface crop harvested from the dataset itself.
3. **Flip the polarity** — ~72% light card + dark glyph, matching the real photo, with a
   contrast floor so the glyph always stays readable.
4. **Match the scene** — the card inherits the original face's lighting gradient (masked blur,
   not a global one), then gets camera-matched blur and sensor noise.

Polygon labels stay valid throughout, since the glyph never moves.


In [ ]:
# --- texture bank + glyph geometry ---------------------------------------
# See section-3 notes above for why each piece exists.
# ---------------------------------------------------------------- texture bank

def _natural_bgr(rng, light=True):
    """A plausible surface colour: any hue, restrained saturation."""
    hsv = np.array([[[rng.integers(0, 180),
                      rng.integers(0, 90),
                      rng.integers(135, 245) if light else rng.integers(25, 105)]]], np.uint8)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR).reshape(3).astype(np.float32)


def _tint(base, field, rng, amp=None):
    """field in [0,1] -> BGR image modulating `base` toward a second colour."""
    other = _natural_bgr(rng, light=bool(rng.integers(0, 2)))
    amp = rng.uniform(0.15, 0.55) if amp is None else amp
    f = (field[..., None] - 0.5) * amp
    return np.clip(base[None, None] * (1 - f) + other[None, None] * f, 0, 255)


def tex_weave(size, rng):
    """Waffle / knit grid -- the most common real card texture."""
    period = rng.uniform(size / 40, size / 8)
    yy, xx = np.mgrid[0:size, 0:size].astype(np.float32)
    if rng.random() < 0.35:                       # diagonal knit
        xx, yy = (xx + yy) / 1.41, (xx - yy) / 1.41
    f = (np.sin(2 * np.pi * xx / period) * np.sin(2 * np.pi * yy / period))
    f = np.abs(f) ** rng.uniform(0.4, 1.6)
    f += rng.normal(0, 0.08, f.shape)
    return _tint(_natural_bgr(rng), np.clip(f, 0, 1), rng)


def tex_ribbed(size, rng):
    """Fine vertical/horizontal ribbing (corduroy, knit columns)."""
    period = rng.uniform(size / 60, size / 12)
    yy, xx = np.mgrid[0:size, 0:size].astype(np.float32)
    ang = rng.uniform(0, np.pi)
    f = np.sin(2 * np.pi * (xx * np.cos(ang) + yy * np.sin(ang)) / period) * 0.5 + 0.5
    f = np.clip(f + rng.normal(0, 0.12, f.shape), 0, 1)
    return _tint(_natural_bgr(rng), f, rng)


def tex_mottle(size, rng):
    """Low-frequency blobs -- wood chip, marble, camo."""
    k = int(rng.integers(3, 12))
    small = rng.random((k, k)).astype(np.float32)
    f = cv2.resize(small, (size, size), interpolation=cv2.INTER_CUBIC)
    f = np.clip((f - f.min()) / (np.ptp(f) + 1e-6), 0, 1)
    if rng.random() < 0.5:                        # sharpen into chip-like flakes
        f = (f > rng.uniform(0.35, 0.65)).astype(np.float32)
        f = cv2.GaussianBlur(f, (0, 0), size / 120)
    return _tint(_natural_bgr(rng), f, rng, amp=rng.uniform(0.3, 0.8))


def tex_glitter(size, rng):
    """Metallic / glitter card: smooth gradient plus fine bright specks."""
    yy, xx = np.mgrid[0:size, 0:size].astype(np.float32)
    ang = rng.uniform(0, 2 * np.pi)
    g = (xx * np.cos(ang) + yy * np.sin(ang))
    g = (g - g.min()) / (np.ptp(g) + 1e-6)
    img = _tint(_natural_bgr(rng), g, rng, amp=rng.uniform(0.3, 0.7))
    spec = (rng.random((size, size)) > rng.uniform(0.90, 0.985)).astype(np.float32)
    spec = cv2.GaussianBlur(spec, (0, 0), rng.uniform(0.5, 1.5))
    return np.clip(img + spec[..., None] * rng.uniform(40, 130), 0, 255)


def tex_photo(size, rng, bank):
    """A real surface crop harvested from the dataset photos themselves."""
    if not bank:
        return tex_mottle(size, rng)
    crop = bank[int(rng.integers(0, len(bank)))]
    out = cv2.resize(crop, (size, size), interpolation=cv2.INTER_LINEAR).astype(np.float32)
    return np.clip(out * rng.uniform(0.7, 1.5), 0, 255)


_TEXTURES = (tex_weave, tex_ribbed, tex_mottle, tex_glitter)


def make_texture(size, rng, bank):
    if bank and rng.random() < 0.35:
        return tex_photo(size, rng, bank)
    return _TEXTURES[int(rng.integers(0, len(_TEXTURES)))](size, rng)


def build_photo_bank(root, read_polys, n_images=150, crop=128, per_image=3, seed=0):
    """Crop real surfaces (floor, wood, tile, wall) from training photos,
    skipping anything that overlaps a labelled glyph."""
    rng = np.random.default_rng(seed)
    lbls = sorted((Path(root) / "train" / "labels").glob("*.txt"))
    idx = rng.permutation(len(lbls))[:n_images]
    bank = []
    for i in idx:
        lbl = lbls[int(i)]
        img = cv2.imread(str(Path(root) / "train" / "images" / (lbl.stem + ".jpg")))
        if img is None:
            continue
        h, w = img.shape[:2]
        occupied = np.zeros((h, w), np.uint8)
        for _, pts in read_polys(lbl, w, h):
            x, y, bw, bh = cv2.boundingRect(pts)
            cv2.rectangle(occupied, (x - bw, y - bh), (x + 2 * bw, y + 2 * bh), 255, -1)
        for _ in range(per_image):
            s = int(rng.integers(crop, min(h, w) // 2 + crop))
            x0, y0 = int(rng.integers(0, w - s)), int(rng.integers(0, h - s))
            if occupied[y0:y0 + s, x0:x0 + s].any():
                continue
            bank.append(cv2.resize(img[y0:y0 + s, x0:x0 + s], (crop, crop)))
    return bank


# ------------------------------------------------------------- glyph geometry

ARROW_NAMES = ("36", "37", "38", "39")     # up / down / right / left arrow


def glyph_mask_with_holes(gray, poly_pts, grow=False):
    """Recover the true glyph pixels from the label polygon.

    The polygon is a single outline, so filling it also fills the counters of
    B/8/9/D. Fix: threshold the luminance *inside* the polygon against the
    surrounding card, and keep only the glyph-toned pixels.

    `grow` handles a second dataset quirk: on the arrow classes the polygon
    traces only the triangular HEAD, so repainting it turns every arrow into a
    bare triangle. When set, the mask is extended along connected components of
    the same tone that the polygon touches -- the shaft comes along. Growth is
    deliberately capped: on a low-contrast card the glyph merges with the card
    and the component would otherwise swallow the whole scene as one blob."""
    h, w = gray.shape
    poly = np.zeros((h, w), np.uint8)
    cv2.fillPoly(poly, [poly_pts], 255)
    x, y, bw, bh = cv2.boundingRect(poly_pts)
    poly_area = int((poly > 0).sum())
    if poly_area == 0 or bw < 4 or bh < 4:
        return poly

    k = max(3, int(0.12 * max(bw, bh)) | 1)
    ring = cv2.dilate(poly, np.ones((k, k), np.uint8)) & ~poly
    if ring.sum() == 0:
        return poly
    inside, outside = gray[poly > 0], gray[ring > 0]
    # Otsu over glyph AND card pixels together. Some polygons are faithful glyph
    # outlines, so their interior is a single tone -- Otsu on the interior alone
    # is unimodal there and slices the glyph in half (this is what turned every
    # 'E' into a stray fragment).
    both = np.concatenate([inside, outside]).reshape(-1, 1).astype(np.uint8)
    thr, _ = cv2.threshold(both, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    bright = inside.mean() > outside.mean()
    sel = ((gray > thr) if bright else (gray < thr)).astype(np.uint8)

    mask = poly & (sel * 255)
    area = mask.sum() / 255
    # < 20%: threshold failed. > 85%: the polygon is already a faithful glyph
    # outline with no counters to carve out, so use it as-is. Either way we
    # fall through to the grow step below rather than returning -- arrow
    # polygons are faithful outlines of the head and still need their shaft.
    if not 0.20 * poly_area < area < 0.85 * poly_area:
        mask = poly.copy()
    else:
        # Otsu eats the anti-aliased stroke edge and leaves thin glyphs
        # skeletal, which desyncs the painted pixels from the mask label.
        # Give one pixel back, clipped to the polygon. This must happen AFTER
        # the guard above: dilating first pushes `area` past 0.85, trips the
        # fallback, and every 6/8/9/A comes out as a filled blob.
        fat = poly & (cv2.dilate(sel, np.ones((3, 3), np.uint8)) * 255)
        if fat.sum() / 255 < 0.92 * poly_area:   # still leaves a counter open
            mask = fat

    if grow:
        pad = int(1.0 * max(bw, bh))
        x0, y0 = max(0, x - pad), max(0, y - pad)
        x1, y1 = min(w, x + bw + pad), min(h, y + bh + pad)
        sub_sel = cv2.morphologyEx(sel[y0:y1, x0:x1], cv2.MORPH_CLOSE,
                                   np.ones((3, 3), np.uint8))
        sub_seed = mask[y0:y1, x0:x1] > 0
        n, lab = cv2.connectedComponents(sub_sel)
        keep = np.zeros_like(sub_sel)
        for i in range(1, n):
            comp = lab == i
            area = int(comp.sum())
            if area > 3 * poly_area:             # leaked into card / floor
                continue
            if (comp & sub_seed).sum() > 0.05 * min(area, poly_area):
                keep |= comp.astype(np.uint8)
        grown = np.zeros((h, w), np.uint8)
        grown[y0:y1, x0:x1] = keep * 255
        grown |= mask
        # an arrow is head + shaft: a sane grow is ~1-3x the head, never more
        if poly_area < grown.sum() / 255 <= 3.5 * poly_area:
            mask = grown
    return mask


def card_quad(glyph_mask, poly_pts, rng):
    """Square card covering the glyph with a real-card margin.

    Axis-aligned plus a few degrees of jitter -- both the training cards and
    the real arena cards sit roughly upright. (minAreaRect's own angle is
    unusable here: on arrow-shaped masks it happily returns 45 degrees and
    every card comes out as a diamond.)"""
    pts = cv2.findNonZero(glyph_mask)
    pts = pts if pts is not None and len(pts) >= 3 else poly_pts.reshape(-1, 1, 2)
    x, y, bw, bh = cv2.boundingRect(pts)
    cx, cy = x + bw / 2, y + bh / 2
    # A real card is square, but a card seen at a steep angle foreshortens and
    # so does the glyph on it. Interpolate between the glyph's own aspect and a
    # square instead of always painting a square over a squashed glyph.
    side = max(bw, bh)
    t = rng.uniform(0.35, 0.85)
    margin = 1.0 + 2 * rng.uniform(0.20, 0.45)
    w_ = (bw + t * (side - bw)) * margin
    h_ = (bh + t * (side - bh)) * margin
    box = cv2.boxPoints(((cx, cy), (w_, h_), rng.uniform(-7, 7)))
    return box.astype(np.float32)


In [ ]:
# --- compositing ----------------------------------------------------------
def randomize_card_background(img, polys, rng, bank=None, tex_size=256,
                              class_names=None):
    """Repaint each labelled card: new textured face + re-coloured glyph,
    lit and blurred to match the surrounding photo. Labels are unchanged.

    polys: list of (class_index, Nx2 int32 polygon), as produced by read_polys.
    class_names: CLASS_NAMES, used only to spot the arrow classes."""
    out = img.astype(np.float32)
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    for cls, pts in polys:
        if cv2.contourArea(pts) < 40:
            continue
        is_arrow = bool(class_names) and class_names[cls] in ARROW_NAMES
        gmask = glyph_mask_with_holes(gray, pts, grow=is_arrow)
        box = card_quad(gmask, pts, rng)

        # --- texture warped onto the card face -------------------------------
        tex = make_texture(tex_size, rng, bank)
        corners = np.float32([[0, 0], [tex_size, 0], [tex_size, tex_size], [0, tex_size]])
        M = cv2.getPerspectiveTransform(corners, box[[1, 2, 3, 0]])
        face = cv2.warpPerspective(tex, M, (w, h), flags=cv2.INTER_LINEAR,
                                   borderMode=cv2.BORDER_REPLICATE)

        card = np.zeros((h, w), np.uint8)
        cv2.fillConvexPoly(card, box.astype(np.int32), 255)
        if card.sum() == 0:
            continue

        # --- inherit the card's own lighting gradient -------------------------
        # Masked (normalised) blur: average ONLY over the original card face,
        # excluding the glyph. A plain global blur bleeds the dark box and
        # bright floor across the card and paints fake diagonal streaks.
        face_px = ((card > 0) & (gmask == 0)).astype(np.float32)
        sigma = max(3.0, 0.20 * max(cv2.boundingRect(box.astype(np.int32))[2:]))
        num = cv2.GaussianBlur(gray.astype(np.float32) * face_px, (0, 0), sigma)
        den = cv2.GaussianBlur(face_px, (0, 0), sigma)
        shade = num / np.maximum(den, 1e-3)
        local = shade[face_px > 0].mean() if face_px.any() else 1.0
        shade = np.clip(shade / max(local, 1e-3), 0.7, 1.3)[..., None]

        light_card = rng.random() < 0.72           # real cards are mostly light
        target = rng.uniform(120, 205) if light_card else rng.uniform(30, 95)
        face_lum = face @ np.float32([0.114, 0.587, 0.299])
        face = np.clip(face * (target / max(face_lum[card > 0].mean(), 1e-3)), 0, 255)
        face *= shade

        # --- glyph colour: opposite polarity, guaranteed contrast -------------
        if light_card:
            glyph = _natural_bgr(rng, light=False) * rng.uniform(0.15, 0.5)
        else:
            glyph = np.clip(_natural_bgr(rng, light=True) * rng.uniform(1.0, 1.3), 0, 255)
        card_lum = (face * np.float32([0.114, 0.587, 0.299]))[card > 0].sum() / max((card > 0).sum(), 1)
        g_lum = float(glyph @ np.float32([0.114, 0.587, 0.299]))
        if abs(card_lum - g_lum) < 60:             # keep the glyph readable
            glyph = np.full(3, 12.0, np.float32) if card_lum > 127 else np.full(3, 243.0, np.float32)

        a_g = cv2.GaussianBlur(gmask.astype(np.float32) / 255, (0, 0), 0.8)[..., None]
        face = face * (1 - a_g) + glyph[None, None] * a_g * shade

        # --- blend the card in, feathered at its edge -------------------------
        a_c = cv2.GaussianBlur(card.astype(np.float32) / 255, (0, 0), 1.0)[..., None]
        out = out * (1 - a_c) + face * a_c

    # match camera softness / sensor noise of the source photo
    if rng.random() < 0.7:
        out = cv2.GaussianBlur(out, (0, 0), rng.uniform(0.4, 1.4))
    out += rng.normal(0, rng.uniform(1.5, 6.0), out.shape)
    return np.clip(out, 0, 255).astype(np.uint8)


In [ ]:
import shutil

SYNTH_FRACTION = 0.5   # randomize backgrounds for 50% of train images
REBUILD = True         # wipe train_synth/ first. Set False only to resume an
                       # interrupted run — leaving it False after changing the
                       # randomizer silently keeps the OLD images, since the
                       # loop below skips any label it has already written.
rng = np.random.default_rng(42)

if REBUILD and (ROOT / "train_synth").exists():
    shutil.rmtree(ROOT / "train_synth")

synth_img_dir = ROOT / "train_synth" / "images"
synth_lbl_dir = ROOT / "train_synth" / "labels"
synth_img_dir.mkdir(parents=True, exist_ok=True)
synth_lbl_dir.mkdir(parents=True, exist_ok=True)

# real surface crops harvested from the dataset's own photos (floor, wood, tile, wall)
PHOTO_BANK = build_photo_bank(ROOT, read_polys, n_images=400)
print(f"photo texture bank: {len(PHOTO_BANK)} crops")

train_lbls = sorted((ROOT / "train" / "labels").glob("*.txt"))
subset = random.Random(42).sample(train_lbls, int(len(train_lbls) * SYNTH_FRACTION))

done = skipped = 0
for lbl in subset:
    out_lbl = synth_lbl_dir / lbl.name
    if out_lbl.exists():
        done += 1
        continue
    img = cv2.imread(str(ROOT / "train" / "images" / (lbl.stem + ".jpg")))
    if img is None:
        continue
    polys = read_polys(lbl, img.shape[1], img.shape[0])
    # The bullseye is a fixed physical target — repainting it would teach a
    # marker that does not exist. Images holding *only* a marker are skipped
    # outright rather than copied through unchanged (that would just duplicate
    # a training image and double its weight).
    polys_to_edit = [(c, p) for c, p in polys if CLASS_NAMES[c] != "marker"]
    if not polys_to_edit:
        skipped += 1
        continue
    synth = randomize_card_background(img, polys_to_edit, rng, PHOTO_BANK,
                                      class_names=CLASS_NAMES)
    cv2.imwrite(str(synth_img_dir / (lbl.stem + ".jpg")), synth)
    out_lbl.write_text(lbl.read_text())
    done += 1
print(f"synthetic images ready: {done}  (skipped {skipped} marker-only)")

# Preview a few, zoomed on the card so the texture and glyph are actually visible
picks = random.Random(7).sample(sorted(synth_img_dir.glob("*.jpg")), 6)
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, p in zip(axes.flat, picks):
    im = cv2.imread(str(p))
    pts = read_polys(synth_lbl_dir / (p.stem + ".txt"), im.shape[1], im.shape[0])[0][1]
    x, y, bw, bh = cv2.boundingRect(pts)
    s = int(max(bw, bh) * 2.8)
    x0, y0 = max(0, x + bw // 2 - s // 2), max(0, y + bh // 2 - s // 2)
    ax.imshow(cv2.cvtColor(im[y0:y0 + s, x0:x0 + s], cv2.COLOR_BGR2RGB)); ax.axis("off")
plt.suptitle("Card-background-randomized training samples"); plt.tight_layout(); plt.show()


## 4. Training

- `yolo26s-seg.pt` pretrained on COCO. Fallback: change one string to `yolo11s-seg.pt`
  (Ultralytics' other production-recommended model) if YOLO26 gives any trouble.
- Train on original **+ synthetic** images; validate/test only on real images.
- **No mirroring** (`fliplr=0, flipud=0`) — orientation-sensitive classes.
- Strong HSV jitter + segment copy-paste + erasing for background/lighting robustness.
- Light geometry only — Roboflow already baked ±4° rotation into the export.

In [ ]:
data_yaml = ROOT / "data_local.yaml"
data_yaml.write_text(
    f"path: {ROOT}\n"
    "train:\n  - train/images\n  - train_synth/images\n"
    "val: valid/images\n"
    "test: test/images\n"
    f"nc: {len(CLASS_NAMES)}\n"
    f"names: {CLASS_NAMES}\n"
)
print(data_yaml.read_text())

In [ ]:
# /dev/shm here is 1 GB and cannot be remounted without root, which caps how many
# dataloader workers can run before they die with "Unexpected bus error ... (shm)".
# Measured peak /dev/shm on this dataset (batch=32, imgsz=640, val enabled):
#     workers=4  -> FAIL, hits 1024/1024 MB      workers=2 -> PASS, 726 MB
#     workers=1  -> PASS, 465 MB
# Worker count dominates, not batch size: Ultralytics builds the validation
# dataloader with its own workers on top of the training one, so 4+4 overruns.
# NB: do NOT "fix" this with set_sharing_strategy("file_system") -- torch puts
# those files in /dev/shm too, so it frees nothing, and it leaks them when a
# worker dies, leaving less shm for every retry (reclaim with:
# `for f in /dev/shm/torch_*; do kill -0 ${f#*torch_} 2>/dev/null || rm -f $f; done`).
# The real fix is a bigger mount: restart the container with `--shm-size=8g`,
# then raise workers back to 8.

model = YOLO("yolo26s-seg.pt")   # fallback: "yolo11s-seg.pt"

results = model.train(
    data=str(data_yaml),
    epochs=60,
    patience=12,          # early stop when val stops improving
    imgsz=640,
    batch=32,
    device=0,
    workers=2,            # measured ceiling for this 1 GB /dev/shm (see above)
    project=str(ROOT / "runs"),
    name="yolo26s_seg_mdp",
    # --- augmentation ---
    fliplr=0.0, flipud=0.0,          # NEVER mirror: left/right arrows, letters, digits
    degrees=5.0, translate=0.1, scale=0.3,
    hsv_h=0.03, hsv_s=0.6, hsv_v=0.5, # strong photometric jitter for lighting robustness
    copy_paste=0.3,                   # segment-based copy-paste onto new contexts
    erasing=0.4,                      # random erasing: robustness to partial occlusion
    mosaic=1.0, close_mosaic=10,
)
BEST = Path(results.save_dir) / "weights" / "best.pt"
print("Best weights:", BEST)


## 5. Evaluation on the untouched test split

In [ ]:
run_dir = Path(results.save_dir)
for name in ("results.png", "confusion_matrix_normalized.png"):
    p = run_dir / name
    if p.exists():
        plt.figure(figsize=(12, 8)); plt.imshow(plt.imread(p)); plt.axis("off"); plt.title(name); plt.show()

In [ ]:
best = YOLO(str(BEST))
test_metrics = best.val(data=str(data_yaml), split="test")
print(f"Box  mAP50: {test_metrics.box.map50:.3f}  mAP50-95: {test_metrics.box.map:.3f}")
print(f"Mask mAP50: {test_metrics.seg.map50:.3f}  mAP50-95: {test_metrics.seg.map:.3f}")

print("\nWeakest classes (box AP50):")
ap = test_metrics.box.ap50
for i in np.argsort(ap)[:5]:
    cls = CLASS_NAMES[test_metrics.box.ap_class_index[i]]
    print(f"  {cls:6s} ({SYMBOL[cls]:11s}) AP50={ap[i]:.3f}")

## 6. Inference — the function the RPi↔PC server calls

Heuristics adopted from the best-documented MDP solution (>95% F1 across cohorts):
1. **Ignore the bullseye** (`marker`) — it is not a scored symbol.
2. Among remaining detections above the confidence floor, pick the **largest bounding box**
   (the target obstacle's symbol is the closest → biggest; a neighbouring obstacle's symbol may
   appear in frame at higher confidence but smaller size).
3. If the path-planning algorithm knows the target is Center/Left/Right in frame, pass `signal=`
   to restrict candidates to that third of the image.

In [ ]:
CONF_THRES = 0.4

def infer(image, model=None, signal=None):
    """image: path or BGR ndarray. signal: None|'C'|'L'|'R' from the algo team.
    Returns (image_id, confidence, annotated ndarray)."""
    model = model or best
    r = model.predict(image, conf=CONF_THRES, imgsz=640, verbose=False)[0]
    cands = []
    for b in r.boxes:
        cls = CLASS_NAMES[int(b.cls)]
        if cls == "marker":
            continue
        x0, y0, x1, y1 = map(float, b.xyxy[0])
        cx = (x0 + x1) / 2 / r.orig_shape[1]
        if signal == "L" and cx > 0.45: continue
        if signal == "R" and cx < 0.55: continue
        if signal == "C" and not (0.25 < cx < 0.75): continue
        cands.append(((x1 - x0) * (y1 - y0), cls, float(b.conf)))
    if not cands:
        return None, 0.0, r.plot()
    _, image_id, conf = max(cands)   # largest area wins
    return image_id, conf, r.plot()

# Demo on random test images
samples = random.Random(1).sample(sorted((ROOT / "test" / "images").glob("*.jpg")), 6)
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, p in zip(axes.flat, samples):
    image_id, conf, annotated = infer(str(p))
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    label = f"ID {image_id} = {SYMBOL[image_id]} ({conf:.2f})" if image_id else "no detection"
    ax.set_title(label); ax.axis("off")
plt.tight_layout(); plt.show()

## 7. Export & next steps

- **PC inference (recommended)**: use `best.pt` directly in the socket server; YOLO26 is
  end-to-end (no NMS), so ONNX export is simpler and faster than older YOLOs.
- **On-RPi inference**: retrain as `yolo26n-seg` and export NCNN/TFLite.
- **Before the real run**: capture a few hundred photos with the actual RPi camera in the
  arena (different lighting/angles/distances, and any new card backgrounds you can see) and
  fine-tune `best.pt` on them for ~10 epochs — the single highest-value step reported by
  previous cohorts.

In [ ]:
from pathlib import Path
from ultralytics import YOLO
BEST = Path("runs/yolo26s_seg_mdp-3/weights/best.pt")
model = YOLO(str(BEST))
model.export(format="ncnn", imgsz=640)

In [ ]:
# Uncomment what you need:
# best.export(format="onnx", imgsz=640)
best.export(format="ncnn", imgsz=640)    # for on-RPi inference
print("Weights for deployment:", BEST)